In [ ]:
import os
import tiktoken

from glob import glob
from PIL import Image

# from rich.pretty import pprint
from pydantic_ai import BinaryContent

from common.cache import RedisCache
from por.llm_agents import (
    ImageDescriber,
    ImageDescriberInput,
    ImageDescriberOutput,
)

In [ ]:
MAX_NUM_TOKENS = 512
tiktoken_encoder = tiktoken.encoding_for_model("gpt-4o")

In [ ]:
def show_image(image_path: str) -> None:
    return Image.open(image_path)

In [ ]:
def get_binary_content(image_path: str) -> BinaryContent:
    with open(image_path, "rb") as image_file:
        return BinaryContent(
            data=image_file.read(),
            media_type="image/jpg",
        )

In [ ]:
def get_description(image_describer_output: ImageDescriberOutput) -> str:
    description = (
        "In the style of GRCRA, an illustration of:\n"
        f"People description: {image_describer_output.people_description}\n"
        f"Scene description: {image_describer_output.scene_description}\n"
    )

    num_tokens = len(tiktoken_encoder.encode(description))
    assert num_tokens <= MAX_NUM_TOKENS

    return description

In [ ]:
image_paths = glob("/resources/cropped-train-images/*.jpg")
print(f"image_paths => {len(image_paths)}")

In [ ]:
description_guidelines = "Carefully analyze the provided image and generate a vivid, highly detailed description."

agent_inputs = [
    ImageDescriberInput(description_guidelines=description_guidelines)
    for _ in image_paths
]

user_contents = [
    get_binary_content(image_path=image_path) for image_path in image_paths
]

image_describer = ImageDescriber(
    cache=RedisCache(),
    max_concurrency=5,
)

image_describer_outputs = await image_describer.batch_generate(
    agent_inputs=agent_inputs,
    user_contents=user_contents,
)

In [ ]:
# idx = 3

# pprint(image_describer_outputs[idx])
# Image.open(image_paths[idx])

In [ ]:
out_base_path = "/resources/cropped-train-images"
for image_path, image_describer_output in zip(
    image_paths,
    image_describer_outputs,
):
    out_file = os.path.basename(image_path).replace(".jpg", ".txt")
    out_path = f"{out_base_path}/{out_file}"
    with open(out_path, "w") as file:
        description = get_description(
            image_describer_output=image_describer_output
        )

        file.write(description)
